In [ ]:
# 1. Imports and Environment Setup
import os
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Set random seeds for reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")


In [ ]:
# 2.1 Download or locate the Kaggle dataset
import kagglehub

try:
    dataset_raw_path = kagglehub.dataset_download('atulyakumar98/gundetection')
    print(f"Dataset successfully downloaded to: {dataset_raw_path}")
except Exception as e:
    # Fallback to local cache if offline or pre-downloaded
    dataset_raw_path = os.path.expanduser(r'~/.cache/kagglehub/datasets/atulyakumar98/gundetection/versions/2')
    print(f"Using local cached dataset path: {dataset_raw_path}")

print(f"Total files in raw directory: {len(os.listdir(dataset_raw_path))}")


In [ ]:
# 2.2 Create Structured Binary Classification Directory (Gun vs No Gun)
BASE_DIR = os.path.abspath("./gun_vs_nogun_data")
os.makedirs(BASE_DIR, exist_ok=True)

for split in ['train', 'val', 'test']:
    for cls_name in ['gun', 'no_gun']:
        os.makedirs(os.path.join(BASE_DIR, split, cls_name), exist_ok=True)

# Select balanced sample size for reliable, efficient training
SAMPLE_PER_CLASS = 600

jpg_files = [f for f in os.listdir(dataset_raw_path) if f.endswith('.jpg')]
random.seed(SEED)
random.shuffle(jpg_files)

gun_samples = []
nogun_samples = []

print("Extracting balanced Gun and No-Gun samples...")
for img_name in jpg_files:
    if len(gun_samples) >= SAMPLE_PER_CLASS and len(nogun_samples) >= SAMPLE_PER_CLASS:
        break
    
    txt_name = os.path.splitext(img_name)[0] + '.txt'
    img_path = os.path.join(dataset_raw_path, img_name)
    txt_path = os.path.join(dataset_raw_path, txt_name)
    
    if not os.path.exists(txt_path):
        continue
        
    try:
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w, _ = img.shape
        if h < 50 or w < 50:
            continue
            
        with open(txt_path, 'r') as fp:
            lines = [l.strip() for l in fp.readlines() if l.strip()]
            
        # Parse bounding boxes
        boxes = []
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                _, xc, yc, bw, bh = map(float, parts[:5])
                # Convert normalized YOLO to pixel coords: x1, y1, x2, y2
                x1 = int((xc - bw / 2.0) * w)
                y1 = int((yc - bh / 2.0) * h)
                x2 = int((xc + bw / 2.0) * w)
                y2 = int((yc + bh / 2.0) * h)
                boxes.append((max(0, x1), max(0, y1), min(w, x2), min(h, y2)))
        
        # 1. Gun sample: The image contains firearms
        if len(gun_samples) < SAMPLE_PER_CLASS:
            gun_samples.append(img_path)
            
        # 2. No Gun sample: Extract non-gun background crop with 0 overlap with gun bounding boxes
        if len(nogun_samples) < SAMPLE_PER_CLASS and boxes:
            # Try to crop a clean negative background patch
            crop_size = min(h // 2, w // 2, 200)
            if crop_size > 40:
                for _ in range(5):
                    cx = random.randint(0, max(1, w - crop_size))
                    cy = random.randint(0, max(1, h - crop_size))
                    c_x2, c_y2 = cx + crop_size, cy + crop_size
                    
                    # Check overlap with any gun box
                    has_overlap = False
                    for (bx1, by1, bx2, by2) in boxes:
                        inter_x1 = max(cx, bx1)
                        inter_y1 = max(cy, by1)
                        inter_x2 = min(c_x2, bx2)
                        inter_y2 = min(c_y2, by2)
                        if inter_x2 > inter_x1 and inter_y2 > inter_y1:
                            has_overlap = True
                            break
                    
                    if not has_overlap:
                        bg_crop = img[cy:c_y2, cx:c_x2]
                        if bg_crop.size > 0:
                            save_name = f"bg_{len(nogun_samples)}_{img_name}"
                            save_path = os.path.join(BASE_DIR, 'train', 'no_gun', save_name)
                            cv2.imwrite(save_path, bg_crop)
                            nogun_samples.append(save_path)
                            break
    except Exception as err:
        continue

print(f"Prepared {len(gun_samples)} Gun candidates and {len(nogun_samples)} No-Gun candidates.")


In [ ]:
# 2.3 Partition into Train (70%), Validation (15%), and Test (15%)
def split_and_copy(sample_list, class_name, is_pre_saved=False):
    train_items, temp_items = train_test_split(sample_list, test_size=0.30, random_state=SEED)
    val_items, test_items = train_test_split(temp_items, test_size=0.50, random_state=SEED)
    
    splits = {'train': train_items, 'val': val_items, 'test': test_items}
    for split_name, items in splits.items():
        dst_folder = os.path.join(BASE_DIR, split_name, class_name)
        for idx, src in enumerate(items):
            dst_file = os.path.join(dst_folder, f"{class_name}_{idx}.jpg")
            if is_pre_saved and src != dst_file and os.path.exists(src):
                shutil.move(src, dst_file)
            elif os.path.exists(src):
                shutil.copy(src, dst_file)

# Copy Gun images to respective splits
split_and_copy(gun_samples[:SAMPLE_PER_CLASS], 'gun', is_pre_saved=False)

# Reorganize No-Gun images
train_bg = [os.path.join(BASE_DIR, 'train', 'no_gun', f) for f in os.listdir(os.path.join(BASE_DIR, 'train', 'no_gun'))]
split_and_copy(train_bg, 'no_gun', is_pre_saved=True)

# Verify split statistics
split_summary = []
for split in ['train', 'val', 'test']:
    gun_count = len(os.listdir(os.path.join(BASE_DIR, split, 'gun')))
    nogun_count = len(os.listdir(os.path.join(BASE_DIR, split, 'no_gun')))
    split_summary.append({'Split': split.capitalize(), 'Gun': gun_count, 'No Gun': nogun_count, 'Total': gun_count + nogun_count})

summary_df = pd.DataFrame(split_summary)
print("=== Dataset Split Summary ===")
print(summary_df.to_string(index=False))


In [ ]:
# 3.1 Visualize Random Samples from Both Classes
plt.figure(figsize=(14, 6))

for i in range(4):
    # Gun sample
    gun_files = os.listdir(os.path.join(BASE_DIR, 'train', 'gun'))
    g_img = Image.open(os.path.join(BASE_DIR, 'train', 'gun', random.choice(gun_files)))
    plt.subplot(2, 4, i + 1)
    plt.imshow(g_img)
    plt.title("Class: Gun", color='darkred', fontweight='bold')
    plt.axis('off')
    
    # No-Gun sample
    nogun_files = os.listdir(os.path.join(BASE_DIR, 'train', 'no_gun'))
    ng_img = Image.open(os.path.join(BASE_DIR, 'train', 'no_gun', random.choice(nogun_files)))
    plt.subplot(2, 4, i + 5)
    plt.imshow(ng_img)
    plt.title("Class: No Gun", color='darkgreen', fontweight='bold')
    plt.axis('off')

plt.suptitle("Sample Images from Training Set (Gun vs No Gun)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 3.2 Dataset Distribution Chart
plt.figure(figsize=(8, 4))
df_melted = summary_df.melt(id_vars=['Split'], value_vars=['Gun', 'No Gun'], var_name='Class', value_name='Count')
sns.barplot(data=df_melted, x='Split', y='Count', hue='Class', palette=['#c0392b', '#27ae60'])
plt.title("Class Distribution Across Train, Validation, and Test Sets", fontsize=12, fontweight='bold')
plt.ylabel("Number of Images")
plt.xlabel("Dataset Split")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title="Class")
plt.show()


In [ ]:
# 4.1 Load Datasets via tf.keras.utils.image_dataset_from_directory
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'test'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

class_names = train_ds.class_names
print(f"Detected Class Names: {class_names}")  # ['gun', 'no_gun']

# 4.2 Data Augmentation Pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

# Optimize input pipeline
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Data pipelines successfully configured with caching & prefetching.")


In [ ]:
# 5.1 Build CNN Model
def build_gun_classifier(input_shape=(128, 128, 3)):
    inputs = layers.Input(shape=input_shape)
    
    # Preprocessing
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)
    
    # Block 1
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Block 2
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Block 3
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Block 4
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Classification Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Gun_Classifier_CNN")
    return model

model = build_gun_classifier()
model.summary()


In [ ]:
# 6.1 Compile Model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

# 6.2 Define Callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

callbacks_list = [early_stopping, reduce_lr]


In [ ]:
# 7.1 Train the CNN Model
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)


In [ ]:
# 8.1 Plot Training & Validation Progression Curves
history_dict = history.history
epochs_range = range(1, len(history_dict['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss Curve
axes[0].plot(epochs_range, history_dict['loss'], 'o-', label='Training Loss', color='#2980b9', linewidth=2)
axes[0].plot(epochs_range, history_dict['val_loss'], 's--', label='Validation Loss', color='#e74c3c', linewidth=2)
axes[0].set_title('Training vs Validation Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# Accuracy Curve
axes[1].plot(epochs_range, history_dict['accuracy'], 'o-', label='Training Accuracy', color='#27ae60', linewidth=2)
axes[1].plot(epochs_range, history_dict['val_accuracy'], 's--', label='Validation Accuracy', color='#f39c12', linewidth=2)
axes[1].set_title('Training vs Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# 8.2 Test Set Evaluation
test_results = model.evaluate(test_ds, verbose=0)
print(f"Test Loss:      {test_results[0]:.4f}")
print(f"Test Accuracy:  {test_results[1] * 100:.2f}%")
print(f"Test Precision: {test_results[2]:.4f}")
print(f"Test Recall:    {test_results[3]:.4f}")

# 8.3 Generate Predictions for Confusion Matrix and ROC
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_pred_probs.extend(probs.flatten())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs >= 0.5).astype(int)

# 8.4 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0],
            annot_kws={"size": 14, "weight": "bold"})
axes[0].set_title("Test Set Confusion Matrix", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# 8.5 ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='#8e44ad', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=12, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# 8.6 Classification Report
print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


In [ ]:
# 9.1 Define Single-Image Prediction Function
def predict_gun_image(image_path, model, threshold=0.5):
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((128, 128))
    img_array = np.array(img_resized, dtype=np.float32)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Predict probability
    raw_pred = model.predict(img_batch, verbose=0)[0][0]
    
    # In binary classification with class_names ['gun', 'no_gun']:
    # label 0 = gun, label 1 = no_gun
    if raw_pred >= threshold:
        predicted_class = class_names[1]  # 'no_gun'
        confidence = raw_pred * 100
        color = 'green'
    else:
        predicted_class = class_names[0]  # 'gun'
        confidence = (1 - raw_pred) * 100
        color = 'red'
        
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Prediction: {predicted_class.upper()}\nConfidence: {confidence:.2f}%", 
              color=color, fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence

# 9.2 Test on Unseen Sample Images from Test Set
test_gun_dir = os.path.join(BASE_DIR, 'test', 'gun')
test_nogun_dir = os.path.join(BASE_DIR, 'test', 'no_gun')

sample_gun = os.path.join(test_gun_dir, os.listdir(test_gun_dir)[0])
sample_nogun = os.path.join(test_nogun_dir, os.listdir(test_nogun_dir)[0])

print("Testing Positive ('Gun') Sample:")
predict_gun_image(sample_gun, model)

print("Testing Negative ('No Gun') Sample:")
predict_gun_image(sample_nogun, model)
